In [ ]:
%run /Users/satyak/iceberg/milvus/class/apikey.ipynb

In [2]:
from langchain import OpenAI
llm = OpenAI() 
llm("Hello world!")


ModuleNotFoundError: No module named 'langchain'

In [ ]:
from typing import List, Dict, Any
from typing import Optional

from langchain_core.tools import tool
import requests

In [ ]:
# Tool 1: Fetch Weather
@tool
def fetch_weather(city: str) -> str:
    """
    Fetch current weather information for a city.
    
    Args:
        city (str): The name of the city to get weather information for.
        
    Returns:
        str: A formatted string containing the weather information.
    """
    return f"The weather in {city}  is sunny with a high of 75°F."

In [ ]:
# Tool 2: Perform Calculation
@tool
def calculate(operation: str, num1: float, num2: float) -> str:
    """
    Perform mathematical calculations between two numbers.
    
    Args:
        operation (str): The mathematical operation to perform. Supported operations: 'add', 'subtract', 'multiply', 'divide'.
        num1 (float): The first number in the calculation.
        num2 (float): The second number in the calculation.
        
    Returns:
        str: A formatted string containing the calculation result or error message.
    """
    if operation == "add":
        return f"The result of adding {num1} and {num2} is {num1 + num2}."
    elif operation == "subtract":
        return f"The result of subtracting {num2} from {num1} is {num1 - num2}."
    elif operation == "multiply":
        return f"The result of multiplying {num1} and {num2} is {num1 * num2}."
    elif operation == "divide" and num2 != 0:
        return f"The result of dividing {num1} by {num2} is {num1 / num2}."
    else:
        return "Invalid operation or division by zero."

In [ ]:
# Tool 3: Time Zone Information
@tool
def get_time_zone(city: str) -> str:
    """
    Get time zone information for a specific city.
    
    Args:
        city (str): The name of the city to get time zone information for. Supported cities: 'New York', 'San Francisco', 'London'.
        
    Returns:
        str: The time zone information for the city, or a message if the city is not supported.
    """
    time_zones = {
        "New York": "Eastern Time (ET)",
        "San Francisco": "Pacific Time (PT)",
        "London": "Greenwich Mean Time (GMT)"
    }
    return time_zones.get(city, "Time zone information not available for this city.")

In [ ]:
import snowflake.connector
import logging
import sys
sys.path.append('/Users/satyak/iceberg-mcp-main')
from iceberg_mcp.config.defaults import DEFAULTS

logger = logging.getLogger(__name__)

@tool
def get_stored_procedure_definition(
        database: str,
        schema: str,
        proc_name: str,
        signature: Optional[str] = None
    ) -> Optional[str]:
        """
        Retrieves the DDL of a stored procedure from Snowflake.

        Args:
            database (str): The database name.
            schema (str): The schema name.
            proc_name (str): The procedure name.
            signature (Optional[str]): The argument signature with parentheses, e.g., '(INT, STRING)'. If not provided, empty parentheses will be appended.

        Returns:
            Optional[str]: The DDL of the stored procedure, or None if not found or on error.
        """
        try:
            # Create Snowflake connection using parameters from defaults.py
            conn = snowflake.connector.connect(
                user=DEFAULTS["SNOWFLAKE_USER"],
                password=DEFAULTS["SNOWFLAKE_PASSWORD"],
                account=DEFAULTS["SNOWFLAKE_ACCOUNT"],
                warehouse=DEFAULTS["SNOWFLAKE_WAREHOUSE"],
                database=database,
                schema=schema,
                role=DEFAULTS["SNOWFLAKE_ROLE"]
            )
            cursor = conn.cursor()
            
            if signature:
                fq_name = f"{database}.{schema}.{proc_name}{signature}"
            else:
                fq_name = f"{database}.{schema}.{proc_name}()"

            logger.info(f"Retrieving DDL for stored procedure: {fq_name}")

            sql = "SELECT GET_DDL('procedure', %s);"
            stmt = cursor.execute(sql, (fq_name,))
            result = stmt.fetchone()

            if result and result[0]:
                return result[0]
            else:
                logger.warning(f"No DDL found for stored procedure: {fq_name}")
                return None

        except Exception as e:
            logger.exception(f"Error retrieving DDL for stored procedure '{fq_name}': {e}")
            return None
        finally:
            if 'cursor' in locals():
                cursor.close()
            if 'conn' in locals():
                conn.close()

@tool
def get_stored_procedures_list(
        database: str,
        schema: str
    ) -> Optional[List[Dict[str, str]]]:
        """
        Retrieves a list of stored procedures from Snowflake.

        Args:
            database (str): The database name.
            schema (str): The schema name.

        Returns:
            Optional[List[Dict[str, str]]]: List of procedures with their details, or None if not found or on error.
        """
        try:
            # Create Snowflake connection using parameters from defaults.py
            conn = snowflake.connector.connect(
                user=DEFAULTS["SNOWFLAKE_USER"],
                password=DEFAULTS["SNOWFLAKE_PASSWORD"],
                account=DEFAULTS["SNOWFLAKE_ACCOUNT"],
                warehouse=DEFAULTS["SNOWFLAKE_WAREHOUSE"],
                database=database,
                schema=schema,
                role=DEFAULTS["SNOWFLAKE_ROLE"]
            )
            cursor = conn.cursor()

            logger.info(f"Retrieving stored procedures list for {database}.{schema}")

            sql = """
            SHOW PROCEDURES IN SCHEMA IDENTIFIER(%s);
            """
            schema_identifier = f"{database}.{schema}"
            stmt = cursor.execute(sql, (schema_identifier,))
            results = stmt.fetchall()

            if results:
                procedures = []
                for row in results:
                    procedure_info = {
                        'name': row[1],
                        'signature': row[8],
                        'language': row[5],
                        'created_on': str(row[0]),
                        'owner': row[6]
                    }
                    procedures.append(procedure_info)
                return procedures
            else:
                logger.warning(f"No stored procedures found in {database}.{schema}")
                return []

        except Exception as e:
            logger.exception(f"Error retrieving stored procedures list for '{database}.{schema}': {e}")
            return None
        finally:
            if 'cursor' in locals():
                cursor.close()
            if 'conn' in locals():
                conn.close()

In [ ]:
@tool
def get_top_data_users() -> Optional[List[Dict[str, Any]]]:
    """
    Generate a report of top 10 customers with highest mobile data usage.
    
    Returns:
        Optional[List[Dict[str, Any]]]: List of dictionaries containing:
            - FullName: Customer name
            - Region: Customer region
            - TotalDataUsageMB: Total data usage in MB
        Returns None if error occurs.
    """
    try:
        # Create Snowflake connection using parameters from defaults.py
        conn = snowflake.connector.connect(
            user=DEFAULTS["SNOWFLAKE_USER"],
            password=DEFAULTS["SNOWFLAKE_PASSWORD"],
            account=DEFAULTS["SNOWFLAKE_ACCOUNT"],
            warehouse=DEFAULTS["SNOWFLAKE_WAREHOUSE"],
            database=DEFAULTS["SNOWFLAKE_DATABASE"],
            role=DEFAULTS["SNOWFLAKE_ROLE"]
        )
        cursor = conn.cursor()
        
        logger.info("Retrieving top 10 data users report")
        
        query = """
        SELECT 
            dc.FullName,
            dc.Region,
            SUM(fu.DataUsedMB) AS TotalDataUsageMB
        FROM ICEBERG_DB.SALES.FACTCUSTOMERUSAGE fu
        JOIN ICEBERG_DB.SALES.DIMCUSTOMER dc ON fu.CustomerID = dc.CustomerID
        GROUP BY dc.FullName, dc.Region
        ORDER BY TotalDataUsageMB DESC
        LIMIT 10;
        """
        
        stmt = cursor.execute(query)
        results = stmt.fetchall()
        
        if results:
            data_users = []
            for row in results:
                user_info = {
                    'FullName': row[0],
                    'Region': row[1],
                    'TotalDataUsageMB': row[2]
                }
                data_users.append(user_info)
            return data_users
        else:
            logger.warning("No data users found")
            return []
            
    except Exception as e:
        logger.exception(f"Error retrieving top data users: {e}")
        return None
    finally:
        if 'cursor' in locals():
            cursor.close()
        if 'conn' in locals():
            conn.close()

In [ ]:
chatllm 

In [ ]:
tools_list=[fetch_weather, calculate, get_time_zone,get_stored_procedure_definition,get_top_data_users,get_stored_procedures_list]

In [ ]:
tools_list

In [ ]:
# Bind multiple tools
chat_llm = chatllm.bind_tools(tools_list)


In [ ]:
messages = [
    {"role": "user", "content": "What's the weather in Seattle tomorrow?"},
    {"role": "user", "content": "Can you add 25.5 and 10.3?"},
    {"role": "user", "content": "What's the time zone for London?"}
]

In [ ]:
# Process each message
for message in messages:
    result = chat_llm.invoke([message])
    print(f"User: {message['content']}")
    print(f"AI: {result.tool_calls}")
    print()

In [ ]:
messages = [
    {"role": "user", "content": "show snowflake stored procedues list"}
]

In [ ]:
# Process each message
for message in messages:
    result = chat_llm.invoke([message])
    print(f"User: {message['content']}")
    print(f"AI: {result.tool_calls}")
    print()

In [ ]:
from langchain.agents import Tool, initialize_agent, AgentType

In [ ]:
# Step 3: Initialize Agent with tools
agent = initialize_agent(
    tools=tools_list,
    llm=chatllm,
    agent=AgentType.OPENAI_FUNCTIONS,
    verbose=True
)

In [ ]:
# Step 4: Run the agent
response = agent.run("show snowflake stored procedures list in iceberg_db database and sales schema")
print("Agent result:", response)

In [ ]:
# Step 4: Run the agent
response = agent.run("show top users")
print("Agent result:", response)